# Codebase Onboarding Companion: Sprint 1 (AI / LangChain team)

This notebook covers three tasks, run on **any public GitHub repo**:

| Task | What it does | Uses an LLM? |
|---|---|---|
| **S1-6** | Detects languages, frameworks and package managers from manifest files | No |
| **S1-7** | Splits files into chunks and embeds them locally (HuggingFace) into a vector store | No |
| **S1-8** | Asks Gemini for a schema-enforced JSON module map (Pydantic + `.with_structured_output()`) | Yes |

**How to run**
1. In Colab, open the key icon (Secrets) and add `GOOGLE_API_KEY` with your Gemini key (or paste it when prompted).
2. Set `REPO_URL` (and optionally `BRANCH`) in the config cell.
3. Runtime -> Run all.

In [ ]:
!pip install -q langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-google-genai sentence-transformers pydantic
# Note: pip may print dependency warnings about preinstalled Colab packages. They can be ignored.

## 0. Config

In [ ]:
import os, getpass

REPO_URL = "https://github.com/psf/requests"   # <-- any public repo
BRANCH = ""                                      # leave empty for the default branch

GEMINI_MODEL = "gemini-2.5-flash"                # change here if Google renames/retires the model
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
MAX_FILE_BYTES = 200_000                         # files bigger than this are skipped for embedding
MAX_CHUNKS = 5000                                # safety cap so huge repos do not run forever

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Gemini API key: ")

## 1. Get the repo (input for the AI tasks)

In the real system the Backend team clones the repo (S1-4) and removes sensitive files (S1-5). Here a shallow clone and a tiny stand-in filter mimic that hand-off so the notebook runs on its own.
Sensitive files stay in the file list (so they appear in the map) but their **content is never embedded**.

In [ ]:
import subprocess, shutil, fnmatch
from pathlib import Path

WORK_DIR = Path("repo_work")

SKIP_DIRS = {".git", "node_modules", "vendor", "dist", "build", "target", "__pycache__",
             ".venv", "venv", ".idea", ".vscode", ".next", ".tox", ".mypy_cache",
             ".pytest_cache", "site-packages"}

# Stand-in for the Backend team's sensitive-file detector (S1-5)
SENSITIVE_PATTERNS = [".env", ".env.*", "*.pem", "*.key", "*.p12", "*.pfx", "*.jks",
                      "*.keystore", "id_rsa*", "id_ed25519*", "credentials*"]

def is_sensitive(name):
    return any(fnmatch.fnmatch(name.lower(), p) for p in SENSITIVE_PATTERNS)

def clone_repo(url, branch, dest):
    if dest.exists():
        shutil.rmtree(dest)
    cmd = ["git", "clone", "--depth", "1", "--single-branch"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [url, str(dest)]
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}   # fail instead of asking for a password
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError("Could not clone the repo. Is it public, and are the URL/branch correct?\n" + r.stderr)

def scan_files(root):
    all_files, safe_files = [], []
    for dirpath, dirs, names in os.walk(root):
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
        for n in names:
            rel = (Path(dirpath) / n).relative_to(root).as_posix()
            all_files.append(rel)
            if not is_sensitive(n):
                safe_files.append(rel)
    return sorted(all_files), sorted(safe_files)

clone_repo(REPO_URL, BRANCH, WORK_DIR)
all_files, safe_files = scan_files(WORK_DIR)
print(f"Cloned {REPO_URL}")
print(f"{len(all_files)} files found, {len(all_files) - len(safe_files)} sensitive file(s) kept out of embedding")

## 2. S1-6: Tech-stack detection (no LLM)

Reads manifest files found anywhere in the repo (so monorepos work) and adds a file-extension count to catch languages that have no manifest.

In [ ]:
import json, re, posixpath
import xml.etree.ElementTree as ET
from collections import Counter
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

EXT_LANG = {
    ".py": "Python", ".js": "JavaScript", ".jsx": "JavaScript", ".mjs": "JavaScript", ".cjs": "JavaScript",
    ".ts": "TypeScript", ".tsx": "TypeScript", ".java": "Java", ".kt": "Kotlin", ".go": "Go",
    ".rs": "Rust", ".rb": "Ruby", ".php": "PHP", ".cs": "C#", ".c": "C", ".h": "C",
    ".cpp": "C++", ".cc": "C++", ".hpp": "C++", ".swift": "Swift", ".scala": "Scala", ".dart": "Dart",
}

# dependency name -> framework. A trailing * means "starts with".
FRAMEWORKS = {
    # JavaScript / TypeScript
    "react": "React", "next": "Next.js", "vue": "Vue", "nuxt": "Nuxt", "@angular/core": "Angular",
    "svelte": "Svelte", "express": "Express", "@nestjs/core": "NestJS", "fastify": "Fastify",
    "koa": "Koa", "vite": "Vite", "electron": "Electron", "react-native": "React Native",
    # Python
    "django": "Django", "flask": "Flask", "fastapi": "FastAPI", "streamlit": "Streamlit",
    "tornado": "Tornado", "scrapy": "Scrapy", "torch": "PyTorch", "tensorflow": "TensorFlow",
    "langchain": "LangChain", "pytest": "pytest",
    # Java / Kotlin
    "spring-boot*": "Spring Boot", "spring-webmvc": "Spring MVC", "quarkus*": "Quarkus",
    "micronaut*": "Micronaut", "hibernate-core": "Hibernate",
    # Go
    "github.com/gin-gonic/gin": "Gin", "github.com/labstack/echo*": "Echo", "github.com/gofiber/fiber*": "Fiber",
    # Rust
    "actix-web": "Actix Web", "rocket": "Rocket", "axum": "Axum", "tokio": "Tokio",
    # PHP
    "laravel/framework": "Laravel", "symfony/framework-bundle": "Symfony",
    # Ruby
    "rails": "Ruby on Rails", "sinatra": "Sinatra",
    # .NET
    "microsoft.aspnetcore*": "ASP.NET Core",
}

def match_frameworks(dep_names):
    found = set()
    for dep in dep_names:
        d = dep.lower()
        for key, fw in FRAMEWORKS.items():
            if d == key or (key.endswith("*") and d.startswith(key[:-1])):
                found.add(fw)
    return sorted(found)

def _strip_ns(root):
    for el in root.iter():
        if isinstance(el.tag, str) and "}" in el.tag:
            el.tag = el.tag.split("}", 1)[1]
    return root

# --- one small parser per manifest type; each returns language, package_manager, deps ---
def parse_package_json(text, rel, files):
    data = json.loads(text)
    deps = list(data.get("dependencies", {})) + list(data.get("devDependencies", {}))
    folder = posixpath.dirname(rel)
    has = lambda n: posixpath.join(folder, n) in files or n in files
    if has("pnpm-lock.yaml"): pm = "pnpm"
    elif has("yarn.lock"): pm = "Yarn"
    elif has("bun.lockb") or has("bun.lock"): pm = "Bun"
    else: pm = "npm"
    lang = "TypeScript" if ("typescript" in deps or has("tsconfig.json")) else "JavaScript"
    return {"language": lang, "package_manager": pm, "deps": deps}

def parse_requirements(text, rel, files):
    deps = []
    for line in text.splitlines():
        line = line.split("#")[0].strip()
        if not line or line.startswith("-"):
            continue
        deps.append(re.split(r"[<>=!~\[;@ ]", line)[0])
    return {"language": "Python", "package_manager": "pip", "deps": deps}

def parse_pyproject(text, rel, files):
    data = tomllib.loads(text)
    deps = [re.split(r"[<>=!~\[;@ ]", d.strip())[0] for d in data.get("project", {}).get("dependencies", [])]
    for group in data.get("project", {}).get("optional-dependencies", {}).values():
        deps += [re.split(r"[<>=!~\[;@ ]", d.strip())[0] for d in group]
    tool = data.get("tool", {})
    deps += [k for k in tool.get("poetry", {}).get("dependencies", {}) if k.lower() != "python"]
    pm = "Poetry" if "poetry" in tool else "PDM" if "pdm" in tool else "uv" if "uv" in tool else "pip"
    return {"language": "Python", "package_manager": pm, "deps": deps}

def parse_pipfile(text, rel, files):
    data = tomllib.loads(text)
    return {"language": "Python", "package_manager": "Pipenv",
            "deps": list(data.get("packages", {})) + list(data.get("dev-packages", {}))}

def parse_pom(text, rel, files):
    root = _strip_ns(ET.fromstring(text))
    return {"language": "Java", "package_manager": "Maven",
            "deps": [e.text.strip() for e in root.iter("artifactId") if e.text]}

def parse_gradle(text, rel, files):
    deps = []
    for m in re.finditer(r"""['"]([\w.\-]+):([\w.\-]+)(?::[^'"]*)?['"]""", text):
        deps += [m.group(2), f"{m.group(1)}:{m.group(2)}"]
    lang = "Kotlin" if rel.endswith(".kts") or "kotlin" in text.lower() else "Java"
    return {"language": lang, "package_manager": "Gradle", "deps": deps}

def parse_go_mod(text, rel, files):
    deps = re.findall(r"^\s*(?:require\s+)?([\w.\-]+\.[\w.\-]+/[\w.\-/]+)\s+v[\w.\-+]+", text, re.M)
    return {"language": "Go", "package_manager": "Go modules", "deps": deps}

def parse_cargo(text, rel, files):
    data = tomllib.loads(text)
    return {"language": "Rust", "package_manager": "Cargo",
            "deps": list(data.get("dependencies", {})) + list(data.get("dev-dependencies", {}))}

def parse_composer(text, rel, files):
    data = json.loads(text)
    return {"language": "PHP", "package_manager": "Composer",
            "deps": list(data.get("require", {})) + list(data.get("require-dev", {}))}

def parse_gemfile(text, rel, files):
    return {"language": "Ruby", "package_manager": "Bundler",
            "deps": re.findall(r"""^\s*gem\s+['"]([^'"]+)['"]""", text, re.M)}

def parse_csproj(text, rel, files):
    root = _strip_ns(ET.fromstring(text))
    return {"language": "C#", "package_manager": "NuGet",
            "deps": [e.get("Include") for e in root.iter("PackageReference") if e.get("Include")]}

def parser_for(name):
    if name == "package.json": return parse_package_json
    if name.startswith("requirements") and name.endswith(".txt"): return parse_requirements
    if name == "pyproject.toml": return parse_pyproject
    if name == "Pipfile": return parse_pipfile
    if name == "pom.xml": return parse_pom
    if name in ("build.gradle", "build.gradle.kts"): return parse_gradle
    if name == "go.mod": return parse_go_mod
    if name == "Cargo.toml": return parse_cargo
    if name == "composer.json": return parse_composer
    if name == "Gemfile": return parse_gemfile
    if name.endswith(".csproj"): return parse_csproj
    return None

def detect_tech_stack(root, all_files, max_manifests=100):
    files, manifests = set(all_files), []
    for rel in all_files:
        parser = parser_for(posixpath.basename(rel))
        if not parser:
            continue
        if len(manifests) >= max_manifests:
            break
        try:
            info = parser((Path(root) / rel).read_text(encoding="utf-8", errors="ignore"), rel, files)
            manifests.append({"path": rel, "language": info["language"],
                              "package_manager": info["package_manager"],
                              "frameworks": match_frameworks(info["deps"])})
        except Exception as e:
            manifests.append({"path": rel, "error": f"could not parse ({type(e).__name__})"})

    ext_counts = Counter(EXT_LANG[Path(f).suffix.lower()] for f in all_files if Path(f).suffix.lower() in EXT_LANG)
    total = sum(ext_counts.values())
    languages = [l for l, c in ext_counts.most_common() if c / total >= 0.05] if total else []
    for m in manifests:
        if m.get("language") and m["language"] not in languages:
            languages.append(m["language"])

    return {
        "languages": languages,
        "frameworks": sorted({fw for m in manifests for fw in m.get("frameworks", [])}),
        "package_managers": sorted({m["package_manager"] for m in manifests if m.get("package_manager")}),
        "language_file_counts": dict(ext_counts.most_common()),
        "manifests": manifests,
    }

tech_stack = detect_tech_stack(WORK_DIR, all_files)
print(json.dumps(tech_stack, indent=2))

## 3. S1-7: Chunking + local embedding scaffold

- Splits files with LangChain's `RecursiveCharacterTextSplitter` (language-aware where LangChain supports the language).
- Embeds locally with a HuggingFace model, so no API calls and no rate limits.
- Stores vectors in Chroma, ready for semantic search in Sprint 2.
- Skips binary files, very large files, lockfiles and minified files.

In [ ]:
import random
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_core.documents import Document

# extension -> LangChain Language name (missing names are skipped automatically)
SPLIT_LANG = {".py": "PYTHON", ".js": "JS", ".jsx": "JS", ".mjs": "JS", ".ts": "TS", ".tsx": "TS",
              ".java": "JAVA", ".kt": "KOTLIN", ".go": "GO", ".rs": "RUST", ".rb": "RUBY",
              ".php": "PHP", ".cs": "CSHARP", ".cpp": "CPP", ".cc": "CPP", ".hpp": "CPP",
              ".c": "C", ".h": "C", ".swift": "SWIFT", ".scala": "SCALA",
              ".md": "MARKDOWN", ".html": "HTML"}

SKIP_NAMES = {"package-lock.json", "yarn.lock", "pnpm-lock.yaml", "poetry.lock", "Pipfile.lock",
              "Cargo.lock", "go.sum", "composer.lock", "Gemfile.lock"}
SKIP_SUFFIXES = (".min.js", ".min.css", ".map", ".svg")

_splitters = {}
def get_splitter(ext):
    lang = getattr(Language, SPLIT_LANG.get(ext, ""), None)
    if lang not in _splitters:
        kwargs = dict(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, add_start_index=True)
        _splitters[lang] = (RecursiveCharacterTextSplitter.from_language(language=lang, **kwargs)
                            if lang else RecursiveCharacterTextSplitter(**kwargs))
    return _splitters[lang]

def read_text_file(path):
    # returns (text, None) or (None, reason_skipped)
    name = path.name
    if name in SKIP_NAMES or name.lower().endswith(SKIP_SUFFIXES):
        return None, "lockfile/minified"
    if path.stat().st_size > MAX_FILE_BYTES:
        return None, "too large"
    with open(path, "rb") as f:
        raw = f.read()
    if b"\x00" in raw[:1024]:
        return None, "binary"
    text = raw.decode("utf-8", errors="ignore")
    return (text, None) if text.strip() else (None, "empty")

def build_chunks(root, files):
    chunks, skipped = [], Counter()
    for rel in files:
        try:
            text, reason = read_text_file(Path(root) / rel)
        except OSError:
            skipped["unreadable"] += 1
            continue
        if reason:
            skipped[reason] += 1
            continue
        ext = Path(rel).suffix.lower()
        chunks += get_splitter(ext).create_documents([text], metadatas=[{"source": rel}])
    return chunks, skipped

chunks, skipped = build_chunks(WORK_DIR, safe_files)
print(f"{len(chunks)} chunks created. Files skipped: {dict(skipped)}")

if len(chunks) > MAX_CHUNKS:
    print(f"Large repo: randomly sampling {MAX_CHUNKS} chunks for this scaffold run.")
    chunks = random.Random(0).sample(chunks, MAX_CHUNKS)

In [ ]:
import re, torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding on:", device)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

PERSIST_DIR = "vector_store"
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

repo_name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
collection = ("repo-" + re.sub(r"[^a-zA-Z0-9_-]", "-", repo_name))[:60].rstrip("-_") or "repo-chunks"

vector_store = Chroma(collection_name=collection, embedding_function=embeddings,
                      persist_directory=PERSIST_DIR,
                      collection_metadata={"embedding_model": EMBED_MODEL})

BATCH = 500
for i in range(0, len(chunks), BATCH):
    vector_store.add_documents(chunks[i:i + BATCH])
    print(f"Embedded {min(i + BATCH, len(chunks))}/{len(chunks)}")

# quick check that search works
for d in vector_store.similarity_search("where does the application start?", k=3):
    print("-", d.metadata["source"])

## 4. S1-8: Schema-enforced LLM output (module map JSON)

`.with_structured_output(ModuleMap)` forces Gemini to return exactly this shape.
The schema is **flat** (a list of modules, no nested/recursive tree) because Gemini handles flat schemas more reliably; the Frontend can build the tree from the `path` fields.
The model receives only a trimmed file tree, the detected stack, and the README start, not the whole codebase.

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# ---------- Response schema (what the Frontend receives) ----------
class ModuleInfo(BaseModel):
    name: str = Field(description="Short human-friendly module name, e.g. 'Authentication'")
    path: str = Field(description="Folder or file path of the module, exactly as it appears in the file tree")
    purpose: str = Field(description="One or two plain-English sentences on what this module does")
    key_files: List[str] = Field(description="Most important file paths in this module, exactly as in the file tree")

class EntryPoint(BaseModel):
    path: str = Field(description="File path of the entry point, exactly as in the file tree")
    reason: str = Field(description="Why this file is where execution or the app starts")

class ModuleMap(BaseModel):
    project_summary: str = Field(description="Two to three sentences describing what the project does")
    modules: List[ModuleInfo] = Field(description="The main modules of the codebase (typically 3 to 12)")
    entry_points: List[EntryPoint] = Field(description="Files where the application starts or is invoked")

# ---------- Compact input for the LLM ----------
def build_tree_summary(files, max_lines=400):
    # deepest view that still fits in max_lines; deeper folders are collapsed into counts
    def render(depth):
        lines, collapsed = [], Counter()
        for f in files:
            parts = f.split("/")
            if len(parts) <= depth:
                lines.append(f)
            else:
                collapsed["/".join(parts[:depth])] += 1
        lines += [f"{k}/ ... ({n} files inside)" for k, n in collapsed.items()]
        return sorted(lines)
    for depth in range(10, 0, -1):
        lines = render(depth)
        if len(lines) <= max_lines:
            return "\n".join(lines)
    return "\n".join(render(1)[:max_lines])

def read_readme(root, files, limit=2000):
    for f in files:
        if "/" not in f and f.lower().startswith("readme"):
            return (Path(root) / f).read_text(encoding="utf-8", errors="ignore")[:limit]
    return "(no README found)"

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a software architecture analyst helping new developers understand a codebase. "
     "Use ONLY the information provided. Every path you output must appear in the file tree below; never invent files."),
    ("human",
     "Detected tech stack:\n{tech_stack}\n\n"
     "README (start):\n{readme}\n\n"
     "File tree (deep folders are collapsed):\n{tree}\n\n"
     "Produce the module map for this repository."),
])

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)
chain = prompt | llm.with_structured_output(ModuleMap)

module_map = chain.invoke({
    "tech_stack": json.dumps({k: tech_stack[k] for k in ("languages", "frameworks", "package_managers")}),
    "readme": read_readme(WORK_DIR, all_files),
    "tree": build_tree_summary(all_files),
})

print(module_map.model_dump_json(indent=2))

## 5. Save the outputs (hand-off JSON for Backend / Frontend)

In [ ]:
with open("tech_stack.json", "w") as f:
    json.dump(tech_stack, f, indent=2)
with open("module_map.json", "w") as f:
    f.write(module_map.model_dump_json(indent=2))
print("Saved tech_stack.json and module_map.json (see the Files panel on the left).")